# Quranic Arabic ASR — Whisper Large V3 Turbo + LoRA

Research/training notebook adapted from the original Kaggle experiment. The uploaded adapter configuration confirms `openai/whisper-large-v3-turbo` as the base model and LoRA rank/alpha of 128. Training outputs have been cleared and credentials removed.

> This notebook is provided as a research artifact, not a guaranteed one-click reproduction. Dataset access, Kaggle paths, package versions, and GPU memory may need adjustment. No verified WER result is claimed in this repository.


In [ ]:
!pip install --upgrade torchao
!pip install pyarabic
!pip install evaluate
!pip install jiwer
!pip install alfanous3
!kaggle datasets download hammaadali/quran-clean-without-araab
!unzip /kaggle/working/quran-clean-without-araab.zip
!pip install yt_dlp
# !pip install transformers --upgrade

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
import tensorflow
import librosa
import transformers
import os
import shutil
import IPython.display as ipd
import warnings
import torchaudio
from datasets import Dataset,Audio,load_dataset
import datasets
from huggingface_hub import list_repo_files, hf_hub_download


### Hugging Face authentication

The dataset and base model used in this notebook are public, so authentication is not required to download them.
To upload a model or processor to your own Hugging Face repository, authenticate using `huggingface-cli login` or `huggingface_hub.login()` and enter a token interactively. Never hard-code access tokens in notebooks.


In [ ]:

repo_id = "tarteel-ai/everyayah"

revision = "main"

directory = "partial-test"

files = list_repo_files(repo_id, revision=revision, repo_type="dataset")


download_folder = "/kaggle/working/Quran_dataset"

os.makedirs(download_folder, exist_ok=True)

files_to_download = [
    'data/test-00000-of-00013-18d88ded65aa7b53.parquet',
    'data/test-00001-of-00013-bcc921915a4a15f3.parquet',
    'data/test-00002-of-00013-b4218deff2d7f268.parquet',
    'data/test-00003-of-00013-eb1ac68e070a388e.parquet',
    'data/test-00004-of-00013-1e7bb5cb0e56549a.parquet',
    'data/test-00005-of-00013-4e0f03a48e0aaac5.parquet',
    'data/test-00006-of-00013-fad04ec308138364.parquet',
]

for file_path in files_to_download:
    local_file_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        revision=revision,
        repo_type="dataset",
        cache_dir=download_folder  # Specify the download folder
    )
    print(f"Downloaded: {file_path} to {local_file_path}")



data_folder = "/kaggle/working/Quran_dataset/datasets--tarteel-ai--everyayah/blobs"

dataset = load_dataset("parquet", data_dir=data_folder, split="train")

new_data=dataset.train_test_split(test_size=0.4)

train_subset = new_data['train'].shuffle(seed=42).select(range(4000))
test_subset = new_data['test'].shuffle(seed=42).select(range(1500))

dataset=datasets.DatasetDict({
    'train':train_subset,
    'test':test_subset
})
dataset=dataset.cast_column("audio",Audio(sampling_rate=16000))

In [ ]:
from transformers import WhisperTokenizer,WhisperProcessor

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3-turbo",task="transcribe",language="ar")
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo", task="transcribe",language="ar")


In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo")

model.generation_config.return_timestamps=True
model.generation_config.is_multilingual=False
model.generation_config.language="ar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

In [ ]:
def less_than_30_s(exampel):
  dur=exampel['duration']
  if dur<=30:
    return True
  else:
    return False

dataset=dataset.filter(less_than_30_s)

In [ ]:
def add_comma(exampel):
    txt=exampel['text']+","
    exampel['text']=txt
    return exampel

In [ ]:
dataset=dataset.map(add_comma)

In [ ]:
def Special_tokenizer(example):
  return processor.tokenizer(example['text'])

dataset=dataset.map(Special_tokenizer,remove_columns=['duration',"text",'reciter'])


In [ ]:
# def Special_tokenizer(example):
#   formatted = f"<|startoftranscript|><|ar|><|transcribe|><|0.00|>{example['text']},<|{example['duration']:.2f}|><|endoftext|>"
#   return processor.tokenizer(formatted,add_special_tokens=False)

# dataset=dataset.map(Special_tokenizer,remove_columns=['duration',"text",'reciter'])


In [ ]:
def audio_tokenizer(example):
  return processor.feature_extractor(example['audio']['array'],sampling_rate=16000)
dataset=dataset.map(audio_tokenizer,remove_columns=['audio'])

In [ ]:
def make_inputs_require_grad(module, input, output):
    output.requires_grad_(True)

model.model.encoder.conv1.register_forward_hook(make_inputs_require_grad)

In [ ]:
from peft import LoraConfig,PeftModel,LoraModel,get_peft_model

a_m={"base_model_class": "WhisperForConditionalGeneration","parent_library": "transformers.models.whisper.modeling_whisper"}

config = LoraConfig(r=128, lora_alpha=128, target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2'], lora_dropout=0.05, bias="lora_only",base_model_name_or_path="openai/whisper-large-v3-turbo",auto_mapping=a_m)

model = get_peft_model(model, config)
model.print_trainable_parameters()

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["input_features"] = batch["input_features"].half()

        # Ensure input_features are 3D (batch_size, num_mel_bins, sequence_length)
        # If it's 4D (batch_size, 1, num_mel_bins, sequence_length), squeeze the extra dimension
        if batch["input_features"].dim() == 4 and batch["input_features"].size(1) == 1:
            batch["input_features"] = batch["input_features"].squeeze(1)

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["input_ids"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch




In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [ ]:
from transformers import Seq2SeqTrainer, TrainerCallback, TrainingArguments, TrainerState, TrainerControl
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR

# This callback helps to save only the adapter weights and remove the base model weights.
class SavePeftModelCallback(TrainerCallback):
    def on_save(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs,
    ):
        checkpoint_folder = os.path.join(args.output_dir, f"{PREFIX_CHECKPOINT_DIR}-{state.global_step}")

        peft_model_path = os.path.join(checkpoint_folder, "adapter_model")
        kwargs["model"].save_pretrained(peft_model_path)

        pytorch_model_path = os.path.join(checkpoint_folder, "pytorch_model.bin")
        if os.path.exists(pytorch_model_path):
            os.remove(pytorch_model_path)
        return control

In [ ]:

from transformers import Seq2SeqTrainingArguments,Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./Quran_text_ASR_model",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_steps=500,
    gradient_checkpointing=True,
    fp16=False,
    eval_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    greater_is_better=False,
    save_strategy="epoch",
    num_train_epochs=1,
    remove_unused_columns=False,  # required as the PeftModel forward doesn't have the signature of the wrapped model's forward
    label_names=["labels"],  # same reason as above

)


In [ ]:
# compute_metrics=compute_metrics
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    callbacks=[SavePeftModelCallback]

)

In [ ]:
trainer.train()


In [ ]:

repo_id = "abosalah/ASR_Quran_model_LoRA_test_version"  

model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)


In [ ]:
from peft import PeftConfig, PeftModel
from transformers import WhisperForConditionalGeneration,WhisperProcessor

base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo")
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo", task="transcribe",language="ar")


In [ ]:
lora_model = PeftModel.from_pretrained(base_model, "abosalah/ASR_Quran_model_LoRA_test_version")
# lora_model=lora_model.merge_and_unload()


In [ ]:
audio=dataset['train'][50]['audio']['array']
dataset['train'][50]['text'],dataset['train'][50]['duration']

In [ ]:

import yt_dlp

url = "https://www.youtube.com/watch?v=yZPrOLhDdm0"

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": "/content/sampels_sound/audio.%(ext)s",
    "noplaylist": True,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

In [ ]:
test_sampel,sr=librosa.load("/content/sampels_sound/audio.webm",sr=16000)

In [ ]:
from transformers import pipeline
import torch

# Load the model with automatic chunking for long audio
pipe = pipeline(
    "automatic-speech-recognition",
    model=lora_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch.float16,
    device=0,
    return_timestamps="word",
    chunk_length_s=30,      # Process in 30-second chunks
    stride_length_s=5,      # 5-second overlap between chunks

)


In [ ]:
result = pipe(test_sampel[:4800000])


In [ ]:
result